# mIF Pipeline: Harpy/SpatialData Assembly Debug Notebook

Use this notebook in the `harpy` environment after merge, InstanSeg, and optional Nimbus have already written file artifacts for the target slide. It demonstrates the explicit base-write then finalize flow against slide-local Nimbus outputs.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "prototyping").exists():
            return candidate
    raise RuntimeError("Could not find the repository root containing src/ and prototyping/.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"=== {label} ===")
    result = func(*args, **kwargs)
    show(result)
    return result

from spatialdata import read_zarr

from mif_pipeline import assemble_spatialdata, finalize_spatialdata, write_spatialdata_base
from mif_pipeline.config import get_slide_config, load_config


In [2]:
CONFIG_PATH = REPO_ROOT / "prototyping" / "prototype_v2-Crop.yaml"
TARGET_SLIDE_ID = "SLIDE-0329_crop_2048"

WRITE_BASE_FORCE = False
FINALIZE_FORCE = False
WRAPPER_FORCE = False
RETURN_SDATA = False
RUN_WRAPPER_TOO = False


In [3]:
config = load_config(CONFIG_PATH)
slide = get_slide_config(config, TARGET_SLIDE_ID)

print("config:", CONFIG_PATH)
print("target slide:", TARGET_SLIDE_ID)
print("full_merge_path:", slide["full_merge"]["ome_path"])
print("cell_mask_path:", Path(slide["mask_export"]["mask_dir"]) / f"{TARGET_SLIDE_ID}{slide['mask_export'].get('suffix', '_whole_cell.tiff')}")
print("nuclear_mask_path:", Path(slide["mask_export"]["mask_dir"]) / f"{TARGET_SLIDE_ID}{slide['mask_export'].get('nuclear_suffix', '_nuclear.tiff')}")
print("nimbus_table_path:", Path(slide["nimbus"]["output_dir"]) / "cell_table_full.csv")
print("spatialdata_store:", slide["spatialdata"]["store_path"])


config: /home/ratnayn/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml
target slide: SLIDE-0329_crop_2048
full_merge_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif
cell_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff
nuclear_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff
nimbus_table_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/cell_table_full.csv
spatialdata_store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr


## Optional Dask Client

If you want to experiment with a Dask client for Harpy operations, start it manually in a separate cell before `finalize_spatialdata(...)`. The pipeline itself does not manage distributed execution.

In [4]:
# from dask.distributed import Client, LocalCluster
# cluster = LocalCluster(n_workers=4, threads_per_worker=1, memory_limit="16GB")
# client = Client(cluster)
# client

## Step 1: Write The Base SpatialData Store

In [5]:
base_results = stage_call(
    f"write_spatialdata_base({TARGET_SLIDE_ID}, dry_run=False)",
    write_spatialdata_base,
    config,
    TARGET_SLIDE_ID,
    force=WRITE_BASE_FORCE,
    return_sdata=False,
)


=== write_spatialdata_base(SLIDE-0329_crop_2048, dry_run=False) ===
[spatialdata] loading image pyramid with tiffslide/zarr: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif
[spatialdata] image_load_seconds completed in 0.16s
[spatialdata] loading label masks for SLIDE-0329_crop_2048
[spatialdata] mask_load_seconds completed in 0.08s
[spatialdata] writing base store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr


/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/ome_zarr/writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(


[spatialdata] write_base_seconds completed in 3.57s
{
  "slide_id": "SLIDE-0329_crop_2048",
  "status": "written",
  "dry_run": false,
  "stage": "write_base",
  "image_loader": "tiffslide_zarr",
  "full_merge_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif",
  "store_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr",
  "cell_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff",
  "nuclear_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff",
  "image_aliases": [
    "R1_CY3_AF",
    "R1_CY5_AF",
    "R1_CY7_AF",
    "R1_FITC_AF",
    "R1_BIT2_RS0584_CY3B",
    "R1_BIT3_RS0015_CY5",
    "R1_BIT4_RS0083_750",
    "R1_DAPI",
    "R1_BIT1_RS0996_488",
    "R2_BIT6_RS0639_CY3B",
    "R2_BIT7_RS0109_CY5",
    "R2_BIT8

## Step 2: Finalize The Same Store

This stage reopens the persisted store, adds aggregation tables, optional Nimbus, and optional shapes, then writes updates back into the canonical slide-local SpatialData store.

In [6]:
finalize_results = stage_call(
    f"finalize_spatialdata({TARGET_SLIDE_ID}, dry_run=False)",
    finalize_spatialdata,
    config,
    TARGET_SLIDE_ID,
    force=FINALIZE_FORCE,
    return_sdata=RETURN_SDATA,
)


=== finalize_spatialdata(SLIDE-0329_crop_2048, dry_run=False) ===
[spatialdata] reopening store for finalize: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr


no parent found for <ome_zarr.reader.Label object at 0x7f114c6e9520>: None
no parent found for <ome_zarr.reader.Label object at 0x7f113562a9c0>: None


[spatialdata] read_store_seconds completed in 0.19s
[spatialdata] deriving shapes from labels: ['cell_labels', 'nuclear_labels']
[spatialdata] shape_derivation_seconds completed in 4.66s
[spatialdata] aggregating intensity tables for ['cell_labels', 'nuclear_labels']


/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/pandas/core/frame.py:6501: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  new_obj.index = new_index
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/pandas/core/frame.py:6501: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  new_obj.index = new_index
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/ratnayn/m

[spatialdata] aggregation_seconds completed in 11.39s
[spatialdata] importing Nimbus table: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/cell_table_full.csv
[spatialdata] nimbus_import_seconds completed in 0.23s
[spatialdata] writing updated elements into store: ['cell_boundaries', 'nuclear_boundaries', 'agg_cell_labels', 'agg_nuclear_labels', 'nimbus_table']


/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/spatialdata/models/models.py:1211: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


[spatialdata] write_elements_seconds completed in 8.62s
[spatialdata] writing updated transformations to store
[spatialdata] write_transformations_seconds completed in 0.33s
{
  "slide_id": "SLIDE-0329_crop_2048",
  "status": "written",
  "dry_run": false,
  "stage": "finalize",
  "store_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr",
  "nimbus_table_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/nimbus/cell_table_full.csv",
  "labels": [
    "cell_labels",
    "nuclear_labels"
  ],
  "shapes": [
    "cell_boundaries",
    "nuclear_boundaries"
  ],
  "tables": [
    "agg_cell_labels",
    "agg_nuclear_labels",
    "nimbus_table"
  ],
  "aggregate": true,
  "aggregate_cell_labels": true,
  "aggregate_nuclear_labels": true,
  "run_on_gpu": true,
  "dask_scheduler": "processes",
  "derive_shapes": true,
  "check_label_overlap": true,
  "load_nimbus": true,
  "nimbus_loaded": true,
  "a

In [7]:
sdata = read_zarr(slide["spatialdata"]["store_path"])
sdata

no parent found for <ome_zarr.reader.Label object at 0x7f11355dacf0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f114f1ac950>: None


SpatialData object, with associated Zarr store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128), (24, 64, 64), (24, 32, 32), (24, 16, 16)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (4895, 4) (2D shapes)
│     └── 'nuclear_boundaries': GeoDataFrame shape: (4339, 4) (2D shapes)
└── Tables
      ├── 'agg_cell_labels': AnnData (4895, 24)
      ├── 'agg_nuclear_labels': AnnData (4339, 24)
      └── 'nimbus_table': AnnData (4895, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels), cell_boundaries (Shapes), nuclear_boundaries (Shapes)

In [8]:
print("images:", list(sdata.images.keys()))
print("labels:", list(sdata.labels.keys()))
print("tables:", list(sdata.tables.keys()))
print("shapes:", list(sdata.shapes.keys()))


images: ['full_image']
labels: ['cell_labels', 'nuclear_labels']
tables: ['agg_cell_labels', 'agg_nuclear_labels', 'nimbus_table']
shapes: ['cell_boundaries', 'nuclear_boundaries']


## Optional Convenience Wrapper

Run this only when you want the one-shot helper after verifying the explicit base/finalize sequence above.

In [ ]:
if RUN_WRAPPER_TOO:
    assemble_results = stage_call(
        f"assemble_spatialdata({TARGET_SLIDE_ID}, dry_run=False)",
        assemble_spatialdata,
        config,
        TARGET_SLIDE_ID,
        force=WRAPPER_FORCE,
        return_sdata=RETURN_SDATA,
    )
